# Spatial EDA Base Template — DuckDB Workflow

**Template ID:** SPATIAL-EDA-BASE  
**Workflow layer:** `raw` (after source → raw spatial ingest)  
**Primary mode:** Notebook + SQL via DuckDB Python API and `spatial` extension

## Purpose

Notebook-first spatial exploratory data analysis template for DuckDB pipelines. Profile geometry quality, extent, and measures on a layer in the **raw**, **staging**, or **curated** layer before cleaning, spatial joins, or export.

Follows the layered workflow convention:

```text
source → raw → staging → curated → output
```

## How to use

1. Copy this notebook into your project (or run it from this repo).
2. Update **Project configuration** with your spatial path, format, table name, and geometry column.
3. Run all cells top to bottom.
4. Record **Findings** and **Next actions** at the end.

**Supported source formats:** Shapefile, GeoJSON, GeoParquet, ESRI File Geodatabase (`.gdb`)  
**Example table:** `raw.raw_parcels` · **Example geometry column:** `geom`  
**Target users:** analysts, data engineers, GIS users, SQL users, Python users.

## 2. Project configuration

Edit the variables below for your spatial dataset. The default example ingests a public California boundary GeoJSON for runnable practice.

| Variable | Purpose |
|----------|----------|
| `SPATIAL_INPUT_PATH` | Path or URL to `.shp`, `.geojson`, `.parquet`, or `.gdb` folder |
| `SPATIAL_TABLE_NAME` | DuckDB table to profile (e.g. `raw.raw_parcels`) |
| `SPATIAL_SOURCE_FORMAT` | `shapefile` · `geojson` · `geoparquet` · `gdb` |
| `GEOMETRY_COLUMN` | Geometry column name (usually `geom`) |
| `OUTPUT_GEOPARQUET_PATH` | Optional export destination |

**ESRI File Geodatabase:** pass the `.gdb` **folder** path and set `GDB_LAYER_NAME`. Availability depends on GDAL drivers bundled with your DuckDB `spatial` extension — see section 5.

In [ ]:
from pathlib import Path

# --- paths ---
PROJECT_ROOT = Path("..").resolve()  # repo root when notebook lives in notebooks/
DB_PATH = PROJECT_ROOT / "work.duckdb"

SPATIAL_INPUT_PATH = (
    "https://raw.githubusercontent.com/glynnbird/usstatesgeojson/master/california.geojson"
)
# Local examples (uncomment and set SPATIAL_SOURCE_FORMAT accordingly):
# SPATIAL_INPUT_PATH = PROJECT_ROOT / "data" / "source" / "parcels.shp"
# SPATIAL_INPUT_PATH = PROJECT_ROOT / "data" / "source" / "parcels.geojson"
# SPATIAL_INPUT_PATH = PROJECT_ROOT / "data" / "source" / "parcels.parquet"
# SPATIAL_INPUT_PATH = PROJECT_ROOT / "data" / "source" / "city_parcels.gdb"

OUTPUT_GEOPARQUET_PATH = PROJECT_ROOT / "data" / "output" / "raw_parcels.parquet"

# --- table and geometry ---
SPATIAL_TABLE_NAME = "raw.raw_parcels"
GEOMETRY_COLUMN = "geom"

# --- source format ---
# Options: shapefile | geojson | geoparquet | gdb
SPATIAL_SOURCE_FORMAT = "geojson"

# --- File Geodatabase only (required when SPATIAL_SOURCE_FORMAT == "gdb") ---
GDB_LAYER_NAME = "Parcels"  # case-sensitive; inspect with ST_Read_Meta first

# --- ingest controls ---
REGISTER_SOURCE = True  # set False when the table already exists
PREVIEW_LIMIT = 20

# --- optional spatial join preview (section 14) ---
RUN_SPATIAL_JOIN_PREVIEW = False
JOIN_TABLE_NAME = "raw.raw_boundary"  # second layer for intersect preview
JOIN_GEOMETRY_COLUMN = "geom"
JOIN_INPUT_PATH = (
    "https://raw.githubusercontent.com/glynnbird/usstatesgeojson/master/california.geojson"
)
REGISTER_JOIN_SOURCE = False  # set True to load JOIN_INPUT_PATH into JOIN_TABLE_NAME

# --- optional GeoParquet export (section 15) ---
EXPORT_GEOPARQUET = False

## 3. Imports

In [ ]:
import duckdb
from IPython.display import display

## 4. Connect to DuckDB

Opens (or creates) a persistent database and ensures workflow schemas exist.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ("raw", "staging", "curated"):
    con.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

OUTPUT_GEOPARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Connected to: {DB_PATH}")
display(con.sql("SELECT version() AS duckdb_version").df())

## 5. Load spatial extension

Install and load `spatial` (required for `ST_Read`, geometry functions, and GeoParquet export). Remote URLs also need `httpfs`.

### ESRI File Geodatabase driver note

FileGDB read support depends on GDAL drivers bundled with your DuckDB build. The usual read driver is **OpenFileGDB**; Esri's proprietary **FileGDB** driver may not be present. Confirm before planning `.gdb` ingest:

```sql
SELECT short_name, long_name, can_open
FROM ST_Drivers()
WHERE short_name IN ('OpenFileGDB', 'FileGDB');
```

If neither driver is available, convert the geodatabase to GeoParquet or Shapefile outside DuckDB, or use a DuckDB build with GDAL FileGDB support. See `docs/03_spatial_ingestion/esri_file_geodatabase.md`.

In [ ]:
EXTENSIONS = ["spatial"]

input_str = str(SPATIAL_INPUT_PATH)
if input_str.startswith(("http://", "https://")):
    EXTENSIONS.insert(0, "httpfs")

for ext in dict.fromkeys(EXTENSIONS):
    con.execute(f"INSTALL {ext};")
    con.execute(f"LOAD {ext};")
    print(f"Loaded extension: {ext}")

if SPATIAL_SOURCE_FORMAT == "gdb":
    print("\n--- GDAL drivers for File Geodatabase ---")
    display(
        con.sql("""
        SELECT short_name, long_name, can_open
        FROM ST_Drivers()
        WHERE short_name IN ('OpenFileGDB', 'FileGDB')
        ORDER BY short_name
        """).df()
    )

## 6. Register spatial source

Materialize the spatial file into the `raw` layer with `ST_Read()`. **Skip this cell** (`REGISTER_SOURCE = False`) when the table already exists from a prior ingest.

| Format | `ST_Read` path | Notes |
|--------|----------------|-------|
| Shapefile | `.shp` file (not the folder) | Requires `.shx` and `.dbf` sidecars |
| GeoJSON | `.geojson` file or URL | Single-layer FeatureCollection |
| GeoParquet | `.parquet` file | Geometry column preserved via GDAL |
| File Geodatabase | `.gdb` folder | Use `layer := 'LayerName'`; names are case-sensitive |

See `docs/03_spatial_ingestion/` and `sql/spatial/ingest_*.sql` for format-specific patterns.

In [ ]:
def resolve_spatial_path(path) -> str:
    path_str = str(path)
    if path_str.startswith(("http://", "https://")):
        return path_str
    return Path(path).resolve().as_posix()


source_path = resolve_spatial_path(SPATIAL_INPUT_PATH)
table_sql = SPATIAL_TABLE_NAME

if REGISTER_SOURCE:
    if SPATIAL_SOURCE_FORMAT == "shapefile":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT *
        FROM ST_Read('{source_path}');
        """
    elif SPATIAL_SOURCE_FORMAT == "geojson":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT *
        FROM ST_Read('{source_path}');
        """
    elif SPATIAL_SOURCE_FORMAT == "geoparquet":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT *
        FROM ST_Read('{source_path}');
        """
    elif SPATIAL_SOURCE_FORMAT == "gdb":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT *
        FROM ST_Read(
          '{source_path}',
          layer := '{GDB_LAYER_NAME}'
        );
        """
    else:
        raise ValueError(
            f"Unsupported SPATIAL_SOURCE_FORMAT: {SPATIAL_SOURCE_FORMAT}. "
            "Use shapefile, geojson, geoparquet, or gdb."
        )

    con.execute(ingest_sql)
    row_count = con.sql(f"SELECT COUNT(*) AS row_count FROM {table_sql}").df().row_count.iloc[0]
    print(f"Registered {SPATIAL_SOURCE_FORMAT} → {SPATIAL_TABLE_NAME} ({row_count:,} rows)")
else:
    print(f"Skipping registration. Profiling existing table: {SPATIAL_TABLE_NAME}")

## 7. Preview attributes

Quick scan of attribute values alongside geometry. Geometry columns may appear as WKB blobs in the DataFrame preview.

In [ ]:
preview_sql = f"""
SELECT *
FROM {SPATIAL_TABLE_NAME}
LIMIT {PREVIEW_LIMIT};
"""

display(con.sql(preview_sql).df())

## 8. Inspect schema

Column names, types, and nullability. Confirm the geometry column name matches `GEOMETRY_COLUMN`.

In [ ]:
schema_sql = f"DESCRIBE {SPATIAL_TABLE_NAME};"
display(con.sql(schema_sql).df())

row_count_sql = f"""
SELECT COUNT(*) AS row_count
FROM {SPATIAL_TABLE_NAME};
"""
print("--- Row count ---")
display(con.sql(row_count_sql).df())

## 9. Geometry type count

Summarize feature counts by `ST_GeometryType` to detect mixed collections or unexpected geometry kinds.

In [ ]:
geom = f'"{GEOMETRY_COLUMN}"'

geom_type_sql = f"""
SELECT
  ST_GeometryType({geom}) AS geometry_type,
  COUNT(*) AS feature_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM {SPATIAL_TABLE_NAME}
WHERE {geom} IS NOT NULL
GROUP BY ST_GeometryType({geom})
ORDER BY feature_count DESC;
"""

display(con.sql(geom_type_sql).df())

## 10. Null geometry check

Quantify `NULL` and empty (`ST_IsEmpty`) geometries before area calculations or spatial joins.

In [ ]:
null_geom_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT({geom}) AS non_null_geom,
  COUNT(*) - COUNT({geom}) AS null_geom,
  ROUND(100.0 * (COUNT(*) - COUNT({geom})) / COUNT(*), 2) AS null_geom_pct,
  SUM(CASE WHEN {geom} IS NOT NULL AND ST_IsEmpty({geom}) THEN 1 ELSE 0 END) AS empty_geom,
  ROUND(
    100.0 * SUM(CASE WHEN {geom} IS NOT NULL AND ST_IsEmpty({geom}) THEN 1 ELSE 0 END) / COUNT(*),
    2
  ) AS empty_geom_pct
FROM {SPATIAL_TABLE_NAME};
"""

display(con.sql(null_geom_sql).df())

## 11. Invalid geometry check

Count and sample topologically invalid geometries. Repair with `ST_MakeValid` in `staging` when needed.

In [ ]:
invalid_summary_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT({geom}) AS with_geom,
  SUM(CASE WHEN {geom} IS NOT NULL AND NOT ST_IsValid({geom}) THEN 1 ELSE 0 END) AS invalid_geom,
  ROUND(
    100.0 * SUM(CASE WHEN {geom} IS NOT NULL AND NOT ST_IsValid({geom}) THEN 1 ELSE 0 END)
      / NULLIF(COUNT({geom}), 0),
    2
  ) AS invalid_geom_pct
FROM {SPATIAL_TABLE_NAME};
"""

print("--- Invalid geometry summary ---")
display(con.sql(invalid_summary_sql).df())

invalid_sample_sql = f"""
SELECT
  ST_GeometryType({geom}) AS geometry_type,
  ST_IsValid({geom}) AS is_valid,
  ST_IsValidReason({geom}) AS invalid_reason
FROM {SPATIAL_TABLE_NAME}
WHERE {geom} IS NOT NULL
  AND NOT ST_IsValid({geom})
LIMIT 20;
"""

invalid_sample = con.sql(invalid_sample_sql).df()
if invalid_sample.empty:
    print("No invalid geometries found.")
else:
    print("--- Sample invalid rows ---")
    display(invalid_sample)

## 12. Spatial extent

Bounding box of the layer — confirm geographic coverage and CRS plausibility before joins.

In [ ]:
extent_sql = f"""
SELECT
  COUNT(*) AS feature_count,
  ST_XMin(ST_Extent({geom})) AS min_x,
  ST_YMin(ST_Extent({geom})) AS min_y,
  ST_XMax(ST_Extent({geom})) AS max_x,
  ST_YMax(ST_Extent({geom})) AS max_y,
  ST_AsText(ST_Extent({geom})) AS extent_wkt
FROM {SPATIAL_TABLE_NAME}
WHERE {geom} IS NOT NULL
  AND NOT ST_IsEmpty({geom});
"""

display(con.sql(extent_sql).df())

## 13. Area or length summary

Summarize polygon area or line length depending on geometry types present. Units follow the layer CRS (degrees² vs meters) — review CRS before interpreting measures. See `docs/05_spatial_eda/crs_check.md`.

In [ ]:
geom_types = con.sql(geom_type_sql).df()["geometry_type"].tolist()
polygon_types = {"POLYGON", "MULTIPOLYGON"}
line_types = {"LINESTRING", "MULTILINESTRING"}

has_polygons = bool(set(geom_types) & polygon_types)
has_lines = bool(set(geom_types) & line_types)

if has_polygons:
    area_sql = f"""
    SELECT
      COUNT(*) AS polygon_count,
      MIN(ST_Area({geom})) AS min_area,
      AVG(ST_Area({geom})) AS avg_area,
      PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ST_Area({geom})) AS median_area,
      MAX(ST_Area({geom})) AS max_area,
      SUM(ST_Area({geom})) AS total_area
    FROM {SPATIAL_TABLE_NAME}
    WHERE {geom} IS NOT NULL
      AND NOT ST_IsEmpty({geom})
      AND ST_GeometryType({geom}) IN ('POLYGON', 'MULTIPOLYGON');
    """
    print("--- Polygon area summary ---")
    display(con.sql(area_sql).df())

if has_lines:
    length_sql = f"""
    SELECT
      COUNT(*) AS line_count,
      MIN(ST_Length({geom})) AS min_length,
      AVG(ST_Length({geom})) AS avg_length,
      PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ST_Length({geom})) AS median_length,
      MAX(ST_Length({geom})) AS max_length,
      SUM(ST_Length({geom})) AS total_length
    FROM {SPATIAL_TABLE_NAME}
    WHERE {geom} IS NOT NULL
      AND NOT ST_IsEmpty({geom})
      AND ST_GeometryType({geom}) IN ('LINESTRING', 'MULTILINESTRING');
    """
    print("--- Line length summary ---")
    display(con.sql(length_sql).df())

if not has_polygons and not has_lines:
    print(
        "No polygon or line geometries detected — skip area/length or add point-density checks."
    )

## 14. Optional spatial join preview

Preview intersect rates between two layers (e.g. parcels vs boundary) before building a full `curated` spatial join. Set `RUN_SPATIAL_JOIN_PREVIEW = True` and configure `JOIN_TABLE_NAME`.

Requires a second table with valid geometries in the same CRS. See `docs/05_spatial_eda/spatial_join_preview.md`.

In [ ]:
join_geom = f'"{JOIN_GEOMETRY_COLUMN}"'

if RUN_SPATIAL_JOIN_PREVIEW:
    if REGISTER_JOIN_SOURCE:
        join_path = resolve_spatial_path(JOIN_INPUT_PATH)
        con.execute(f"""
        CREATE OR REPLACE TABLE {JOIN_TABLE_NAME} AS
        SELECT * FROM ST_Read('{join_path}');
        """)
        print(f"Registered join source → {JOIN_TABLE_NAME}")

    join_preview_sql = f"""
    SELECT
      COUNT(*) AS primary_count,
      SUM(CASE WHEN ST_Intersects(p.{join_geom}, j.{join_geom}) THEN 1 ELSE 0 END) AS intersecting_count,
      ROUND(
        100.0 * SUM(CASE WHEN ST_Intersects(p.{join_geom}, j.{join_geom}) THEN 1 ELSE 0 END) / COUNT(*),
        2
      ) AS pct_intersecting
    FROM {SPATIAL_TABLE_NAME} p
    CROSS JOIN {JOIN_TABLE_NAME} j
    WHERE p.{geom} IS NOT NULL
      AND j.{join_geom} IS NOT NULL
      AND NOT ST_IsEmpty(p.{geom})
      AND NOT ST_IsEmpty(j.{join_geom});
    """
    display(con.sql(join_preview_sql).df())
else:
    print(
        "Skipping spatial join preview. Set RUN_SPATIAL_JOIN_PREVIEW = True "
        "and configure JOIN_TABLE_NAME to enable."
    )

## 15. Export optional GeoParquet

Write the profiled layer to GeoParquet in the `output` layer. Set `EXPORT_GEOPARQUET = True` to run. See `sql/export/export_geoparquet.sql` and `docs/10_export/geoparquet_export.md`.

In [ ]:
if EXPORT_GEOPARQUET:
    output_path = OUTPUT_GEOPARQUET_PATH.resolve().as_posix()
    export_sql = f"""
    COPY (
      SELECT *
      FROM {SPATIAL_TABLE_NAME}
    ) TO '{output_path}'
    (FORMAT GDAL, DRIVER 'Parquet');
    """
    con.execute(export_sql)

    file_rows = con.sql(f"SELECT COUNT(*) AS row_count FROM ST_Read('{output_path}')").df()
    table_rows = con.sql(f"SELECT COUNT(*) AS row_count FROM {SPATIAL_TABLE_NAME}").df()

    print(f"Exported → {OUTPUT_GEOPARQUET_PATH}")
    print(f"Table rows: {table_rows.row_count.iloc[0]:,}")
    print(f"File rows:  {file_rows.row_count.iloc[0]:,}")
else:
    print(
        f"Skipping GeoParquet export. Set EXPORT_GEOPARQUET = True to write "
        f"{OUTPUT_GEOPARQUET_PATH}"
    )

## 16. Findings

Document what you learned during spatial EDA. Replace the bullets below with your observations.

- **Source format:** _e.g. GeoJSON from open-data portal; ingested via `ST_Read`_ 
- **Row volume:** _e.g. 12,475 parcel features — matches source metadata_
- **Geometry types:** _e.g. all `MULTIPOLYGON`; no mixed collections_
- **Null / empty geometry:** _e.g. 0 null; 3 empty — quarantine in staging_
- **Invalid geometry:** _e.g. 38 invalid (0.3%) — plan `ST_MakeValid` in staging_
- **Spatial extent:** _e.g. xmin/ymin/xmax/ymax plausible for county; CRS appears EPSG:4326_
- **Area / length:** _e.g. median parcel area reasonable; check units if CRS is geographic_
- **Spatial join preview:** _e.g. 98% parcels intersect boundary; 2% orphans to investigate_
- **File Geodatabase (if used):** _e.g. OpenFileGDB driver available; layer `Parcels` loaded OK_

## 17. Next actions

Map findings to the next workflow step. Check the boxes that apply to your project.

- [ ] **CRS review** — confirm and harmonize CRS across layers (`docs/05_spatial_eda/crs_check.md`)
- [ ] **Geometry cleaning** — repair invalid geometries, drop empties (`docs/06_cleaning/spatial_geometry_cleaning.md`)
- [ ] **Staging build** — promote `raw` → `staging` with typed attributes and valid geometry (`docs/08_spatial_transformation/`)
- [ ] **Spatial join** — build curated overlay or point-in-polygon models (`docs/08_spatial_transformation/spatial_join.md`)
- [ ] **Validation** — spatial validity gates before export (`docs/09_validation/spatial_validity_check.md`)
- [ ] **Export** — deliver GeoParquet or GeoJSON to `output` (`docs/10_export/geoparquet_export.md`)
- [ ] **Performance** — subset large sources with `spatial_filter_box` (`docs/11_performance/spatial_performance.md`)

---

**Related templates:** `00_eda_base.ipynb` (tabular EDA) · `01_etl_base.ipynb` (layered ETL)  
**Related docs:** `docs/03_spatial_ingestion/` · `docs/05_spatial_eda/` · `sql/spatial/`